# MDAV - Train the AIForge_MDAV diffusion-forgery detector

This notebook trains `best_diffusion.pth`, the AI-generated forgery branch used by `backend/app/services/diffusion_service.py`.

It intentionally follows the same model family as the DocTamper visual branch:

- **Input A:** RGB document crop/image normalized with ImageNet statistics.
- **Input B:** JPEG DCT coefficient map made from the same pixels.
- **Model:** ResNet18 U-Net from `segmentation_models_pytorch`, with a small DCT embedding concatenated to RGB channels.
- **Output:** 2-class pixel mask: class `0` = authentic/background, class `1` = AI-edited/tampered.
- **Checkpoint:** saved as `best_diffusion.pth` with a `{"model": state_dict, ...}` payload that the backend can load directly.

The dataset expected here is `AIForge_MDAV`, containing:

```text
AIForge_MDAV/
  images/          forged document images
  masks/           binary tamper masks, 0 authentic and 255 tampered
  annotations/     unified JSON annotations
  metadata.csv     one row per forged image
```

Attach `AIForge_MDAV` as a Kaggle input dataset, then run this notebook from top to bottom.

## 1. Kaggle setup

Run this first in a fresh Kaggle notebook with GPU enabled. A single T4 is enough for the default settings. If Kaggle gives `GPU T4 x2`, this notebook still uses one GPU by default because the DCT preprocessing is CPU-heavy and a simple single-GPU run is easier to reproduce.

The `FAST_DEV_RUN` switch lets you test the whole notebook quickly before committing to a full training run.

In [ ]:
# Kaggle normally already has torch/torchvision installed. These are the extra packages used here.
!pip install -q jpegio opencv-python-headless six segmentation-models-pytorch==0.3.4 timm==0.9.7 efficientnet-pytorch==0.7.1 pretrainedmodels==0.7.4

In [ ]:
import json
import math
import os
import random
import tempfile
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF

import segmentation_models_pytorch as smp

try:
    import jpegio
except Exception as exc:
    raise RuntimeError("jpegio is required because the backend model contract includes JPEG DCT coefficients") from exc

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
@dataclass
class CFG:
    seed: int = 42
    image_size: int = 512
    batch_size: int = 4
    num_workers: int = 2
    epochs: int = 18
    lr: float = 2e-4
    weight_decay: float = 1e-4
    positive_class_weight: float = 6.0
    tamper_center_crop_prob: float = 0.75
    val_fraction: float = 0.15
    max_records: int | None = None          # set to a number like 500 for a smaller experiment
    fast_dev_run: bool = False              # set True to test all cells in a few minutes
    output_dir: str = "/kaggle/working/aiforge_diffusion_training"
    checkpoint_name: str = "best_diffusion.pth"

cfg = CFG()

if cfg.fast_dev_run:
    cfg.epochs = 1
    cfg.max_records = 128
    cfg.batch_size = 2

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path(cfg.output_dir)
OUT.mkdir(parents=True, exist_ok=True)
print(cfg)
print("Output directory:", OUT)

## 2. Locate and inspect `AIForge_MDAV`

Kaggle mounts input datasets under `/kaggle/input/<dataset-name>/...`. The helper below searches common layouts so the notebook works whether the dataset root is exactly `AIForge_MDAV` or nested one folder deeper.

The metadata CSV was produced on Windows in some runs, so paths may contain backslashes. The notebook normalizes every metadata path before opening files on Linux.

In [ ]:
def normalize_rel_path(value) -> Path:
    text = str(value).strip().replace("\\", "/")
    while text.startswith("./"):
        text = text[2:]
    return Path(text)


def find_aiforge_root() -> Path:
    search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd(), Path.cwd().parent]
    seen = set()
    candidates = []

    def add_candidate(path: Path) -> None:
        try:
            resolved = path.resolve()
        except Exception:
            resolved = path
        key = str(resolved)
        if key not in seen:
            seen.add(key)
            candidates.append(path)

    for base in search_roots:
        if not base.exists():
            continue
        for pattern in ["AIForge_MDAV", "*/AIForge_MDAV", "*/*/AIForge_MDAV", "**/AIForge_MDAV"]:
            for path in base.glob(pattern):
                add_candidate(path)
        for meta in base.glob("**/metadata.csv"):
            add_candidate(meta.parent)

    for root in candidates:
        if (root / "metadata.csv").exists() and (root / "images").exists() and (root / "masks").exists():
            return root.resolve()

    inspected = [str(base) for base in search_roots if base.exists()]
    raise FileNotFoundError(
        "Could not find AIForge_MDAV. Searched under: " + ", ".join(inspected) +
        ". Make sure the package was extracted or the dataset is attached."
    )

DATA_ROOT = find_aiforge_root()
META_PATH = DATA_ROOT / "metadata.csv"
print("AIForge_MDAV root:", DATA_ROOT)
print("Metadata:", META_PATH)

metadata = pd.read_csv(META_PATH)
print("Rows:", len(metadata))
display(metadata.head(3))
print(metadata[["dataset", "tamper_type", "generator", "processing_status"]].describe(include="all"))

In [ ]:
def resolve_dataset_path(rel_value) -> Path:
    rel = normalize_rel_path(rel_value)
    candidates = [DATA_ROOT / rel, DATA_ROOT.parent / rel]
    for p in candidates:
        if p.exists():
            return p
    return DATA_ROOT / rel

rows = metadata.copy()
rows["forged_abs"] = rows["forged_image"].apply(resolve_dataset_path)
rows["mask_abs"] = rows["mask_path"].apply(resolve_dataset_path)
rows["source_abs"] = rows["source_image"].apply(resolve_dataset_path) if "source_image" in rows else None
rows["forged_exists"] = rows["forged_abs"].apply(lambda p: Path(p).exists())
rows["mask_exists"] = rows["mask_abs"].apply(lambda p: Path(p).exists())

missing = rows[(~rows["forged_exists"]) | (~rows["mask_exists"])]
print("Usable forged+mask rows:", len(rows) - len(missing), "/", len(rows))
if len(missing):
    display(missing[["image_id", "forged_image", "mask_path", "forged_exists", "mask_exists"]].head())

rows = rows[rows["forged_exists"] & rows["mask_exists"]].reset_index(drop=True)
if cfg.max_records is not None:
    rows = rows.sample(n=min(cfg.max_records, len(rows)), random_state=cfg.seed).reset_index(drop=True)
print("Training candidate rows after optional cap:", len(rows))
print("Dataset counts:")
print(rows["dataset"].value_counts())

## 3. Optional clean negatives

Each forged image already contains many authentic/background pixels, so pixel-level segmentation training can work with forged images plus their masks alone.

If authentic source images are present in the same Kaggle input, the notebook will add them as **clean negative documents** with an all-zero mask. This improves document-level false-positive calibration. If no authentic source files are present, training still proceeds and the notebook prints a warning instead of inventing fake clean negatives.

In [ ]:
def build_records(df: pd.DataFrame) -> list[dict]:
    records: list[dict] = []
    for _, row in df.iterrows():
        records.append({
            "image_id": str(row["image_id"]),
            "image_path": Path(row["forged_abs"]),
            "mask_path": Path(row["mask_abs"]),
            "is_clean": False,
            "dataset": str(row.get("dataset", "unknown")),
        })

    clean_seen = set()
    if "source_abs" in df.columns:
        for _, row in df.iterrows():
            p = Path(row["source_abs"])
            if p.exists() and p not in clean_seen:
                clean_seen.add(p)
                records.append({
                    "image_id": f"clean_{p.stem}",
                    "image_path": p,
                    "mask_path": None,
                    "is_clean": True,
                    "dataset": str(row.get("dataset", "unknown")),
                })
    return records

records = build_records(rows)
num_clean = sum(r["is_clean"] for r in records)
print("Total records:", len(records))
print("Forged positives:", len(records) - num_clean)
print("Clean authentic negatives found:", num_clean)
if num_clean == 0:
    print("WARNING: no authentic source images were found. Training will use forged images and their zero-mask background pixels only.")

In [ ]:
# Deterministic split by image_id so repeated runs use the same validation set.
records = sorted(records, key=lambda r: r["image_id"])
rng = random.Random(cfg.seed)
rng.shuffle(records)
val_count = max(1, int(len(records) * cfg.val_fraction))
val_records = records[:val_count]
train_records = records[val_count:]
print("Train records:", len(train_records))
print("Val records:", len(val_records))
print("First train record:", train_records[0])

## 4. Preprocessing: what DCT means here

The backend does not feed only RGB pixels into the model. It also computes a JPEG DCT coefficient map:

1. Convert the document image to grayscale.
2. Save it as JPEG at quality 95.
3. Read the JPEG coefficient array with `jpegio`.
4. Take absolute coefficient values and clip them into integer bins `0..20`.
5. Embed those bins with a learned embedding layer, then concatenate that embedding with RGB channels.

Why this helps: many document forgeries leave subtle compression-frequency artifacts around edited text. RGB captures what the eye sees; DCT gives the model a second view of compression/noise consistency. The notebook uses the same constants as `diffusion_service.py` so the saved checkpoint matches backend inference.

In [ ]:
JPEG_QUALITY = 95
DCT_BINS = 21
DCT_DIM = 16
STRIDE = 32
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]


def pad_image_and_mask(image: Image.Image, mask: Image.Image, min_size: int) -> tuple[Image.Image, Image.Image]:
    w, h = image.size
    pad_w = max(0, min_size - w)
    pad_h = max(0, min_size - h)
    if pad_w == 0 and pad_h == 0:
        return image, mask
    left = pad_w // 2
    right = pad_w - left
    top = pad_h // 2
    bottom = pad_h - top
    image = ImageOps.expand(image, (left, top, right, bottom), fill=(255, 255, 255))
    mask = ImageOps.expand(mask, (left, top, right, bottom), fill=0)
    return image, mask


def choose_crop_box(mask_np: np.ndarray, image_size: int, training: bool, tamper_prob: float) -> tuple[int, int, int, int]:
    h, w = mask_np.shape
    if training and mask_np.max() > 0 and random.random() < tamper_prob:
        ys, xs = np.where(mask_np > 127)
        i = random.randrange(len(xs))
        cx, cy = int(xs[i]), int(ys[i])
        jitter = image_size // 5
        cx += random.randint(-jitter, jitter)
        cy += random.randint(-jitter, jitter)
        left = max(0, min(w - image_size, cx - image_size // 2))
        top = max(0, min(h - image_size, cy - image_size // 2))
    elif training:
        left = random.randint(0, max(0, w - image_size))
        top = random.randint(0, max(0, h - image_size))
    else:
        left = max(0, (w - image_size) // 2)
        top = max(0, (h - image_size) // 2)
    return left, top, left + image_size, top + image_size


def rgb_and_dct_from_pil(image: Image.Image) -> tuple[torch.Tensor, torch.Tensor]:
    # Match backend behavior: grayscale -> temporary JPEG -> RGB from JPEG + DCT coefficients from JPEG.
    gray = image.convert("L")
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        gray.save(tmp_path, "JPEG", quality=JPEG_QUALITY)
        jpg = jpegio.read(tmp_path)
        dct = np.clip(np.abs(jpg.coef_arrays[0]), 0, DCT_BINS - 1).astype(np.int64)
        jpeg_rgb = Image.open(tmp_path).convert("RGB")
        jpeg_rgb.load()
    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass

    h = min(dct.shape[0], jpeg_rgb.size[1]) // 8 * 8
    w = min(dct.shape[1], jpeg_rgb.size[0]) // 8 * 8
    jpeg_rgb = jpeg_rgb.crop((0, 0, w, h))
    dct = dct[:h, :w]
    rgb_t = TF.normalize(TF.to_tensor(jpeg_rgb), mean=MEAN, std=STD)
    dct_t = torch.from_numpy(np.ascontiguousarray(dct)).long()
    return rgb_t, dct_t

In [ ]:
class AIForgeryMaskDataset(Dataset):
    def __init__(self, records: list[dict], image_size: int, training: bool):
        self.records = records
        self.image_size = image_size
        self.training = training

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        image = Image.open(rec["image_path"]).convert("RGB")
        if rec["mask_path"] is None:
            mask = Image.new("L", image.size, 0)
        else:
            mask = Image.open(rec["mask_path"]).convert("L")
            if mask.size != image.size:
                mask = mask.resize(image.size, Image.NEAREST)

        image, mask = pad_image_and_mask(image, mask, self.image_size)
        mask_np = np.array(mask)
        crop_box = choose_crop_box(mask_np, self.image_size, self.training, cfg.tamper_center_crop_prob)
        image = image.crop(crop_box)
        mask = mask.crop(crop_box)

        if self.training and random.random() < 0.5:
            image = ImageOps.mirror(image)
            mask = ImageOps.mirror(mask)

        rgb_t, dct_t = rgb_and_dct_from_pil(image)
        mask_np = (np.array(mask)[: rgb_t.shape[1], : rgb_t.shape[2]] > 127).astype(np.int64)
        mask_t = torch.from_numpy(np.ascontiguousarray(mask_np)).long()
        return rgb_t, dct_t, mask_t, rec["image_id"]

train_ds = AIForgeryMaskDataset(train_records, cfg.image_size, training=True)
val_ds = AIForgeryMaskDataset(val_records, cfg.image_size, training=False)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)

sample_rgb, sample_dct, sample_mask, sample_id = train_ds[0]
print("Sample:", sample_id)
print("RGB tensor:", tuple(sample_rgb.shape), sample_rgb.dtype)
print("DCT tensor:", tuple(sample_dct.shape), sample_dct.dtype, int(sample_dct.min()), int(sample_dct.max()))
print("Mask tensor:", tuple(sample_mask.shape), sample_mask.dtype, int(sample_mask.max()))

## 5. Model, loss, and metrics

The tampered text area is tiny compared with the full page, so ordinary cross entropy can learn to predict only background. The loss combines:

- weighted cross entropy, giving class `1` more importance;
- foreground Dice loss, which rewards overlap with the edited mask.

Validation reports pixel-level precision, recall, F1, Dice, and IoU. The backend later aggregates a probability map into a document-level AI-forgery score.

In [ ]:
class DCTEmbed(nn.Module):
    def __init__(self, n_bins: int = DCT_BINS, dim: int = DCT_DIM):
        super().__init__()
        self.emb = nn.Embedding(n_bins, dim)

    def forward(self, dct: torch.Tensor) -> torch.Tensor:
        return self.emb(dct.long()).permute(0, 3, 1, 2).contiguous()


class MDAVNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.dct = DCTEmbed()
        self.net = smp.Unet(
            encoder_name="resnet18",
            encoder_weights=None,
            in_channels=3 + DCT_DIM,
            classes=2,
        )

    def forward(self, image: torch.Tensor, dct: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([image, self.dct(dct)], dim=1))


class CrossEntropyDiceLoss(nn.Module):
    def __init__(self, positive_weight: float):
        super().__init__()
        self.register_buffer("weights", torch.tensor([1.0, positive_weight], dtype=torch.float32))

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, target, weight=self.weights.to(logits.device))
        prob = torch.softmax(logits, dim=1)[:, 1]
        target_f = target.float()
        inter = (prob * target_f).sum(dim=(1, 2))
        denom = prob.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2))
        dice = 1.0 - ((2.0 * inter + 1.0) / (denom + 1.0)).mean()
        return ce + dice


def batch_metrics(logits: torch.Tensor, target: torch.Tensor, threshold: float = 0.5) -> dict[str, float]:
    prob = torch.softmax(logits, dim=1)[:, 1]
    pred = prob >= threshold
    truth = target.bool()
    tp = (pred & truth).sum().item()
    fp = (pred & ~truth).sum().item()
    fn = (~pred & truth).sum().item()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
    return {"precision": precision, "recall": recall, "f1": f1, "iou": iou, "dice": dice}

model = MDAVNet().to(DEVICE)
criterion = CrossEntropyDiceLoss(cfg.positive_class_weight).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.epochs))
scaler = GradScaler(enabled=(DEVICE.type == "cuda"))
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

In [ ]:
def train_one_epoch(epoch: int) -> dict[str, float]:
    model.train()
    total_loss = 0.0
    n_batches = 0
    running = {"precision": 0.0, "recall": 0.0, "f1": 0.0, "iou": 0.0, "dice": 0.0}
    start = time.time()

    for rgb, dct, mask, _ids in train_loader:
        rgb = rgb.to(DEVICE, non_blocking=True)
        dct = dct.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(DEVICE.type == "cuda")):
            logits = model(rgb, dct)
            loss = criterion(logits, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += float(loss.detach().cpu())
        metrics = batch_metrics(logits.detach(), mask)
        for k, v in metrics.items():
            running[k] += v
        n_batches += 1

    out = {"loss": total_loss / max(1, n_batches), "seconds": time.time() - start}
    out.update({k: v / max(1, n_batches) for k, v in running.items()})
    print(f"epoch {epoch:02d} train loss={out['loss']:.4f} dice={out['dice']:.4f} f1={out['f1']:.4f} time={out['seconds']:.1f}s")
    return out


@torch.no_grad()
def validate(epoch: int) -> dict[str, float]:
    model.eval()
    total_loss = 0.0
    n_batches = 0
    running = {"precision": 0.0, "recall": 0.0, "f1": 0.0, "iou": 0.0, "dice": 0.0}

    for rgb, dct, mask, _ids in val_loader:
        rgb = rgb.to(DEVICE, non_blocking=True)
        dct = dct.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        logits = model(rgb, dct)
        loss = criterion(logits, mask)
        total_loss += float(loss.detach().cpu())
        metrics = batch_metrics(logits, mask)
        for k, v in metrics.items():
            running[k] += v
        n_batches += 1

    out = {"loss": total_loss / max(1, n_batches)}
    out.update({k: v / max(1, n_batches) for k, v in running.items()})
    print(f"epoch {epoch:02d} valid loss={out['loss']:.4f} dice={out['dice']:.4f} f1={out['f1']:.4f} iou={out['iou']:.4f}")
    return out

## 6. Train and save `best_diffusion.pth`

The best checkpoint is selected by validation Dice. For a real run, leave `FAST_DEV_RUN=False` and let the notebook train for the configured number of epochs. For a quick plumbing test, set `fast_dev_run=True` in the config cell.

In [ ]:
best_score = -1.0
history = []
best_path = OUT / cfg.checkpoint_name

for epoch in range(1, cfg.epochs + 1):
    train_metrics = train_one_epoch(epoch)
    val_metrics = validate(epoch)
    scheduler.step()
    row = {"epoch": epoch, "train": train_metrics, "val": val_metrics, "lr": scheduler.get_last_lr()[0]}
    history.append(row)
    score = val_metrics["dice"]
    if score > best_score:
        best_score = score
        payload = {
            "model": model.state_dict(),
            "config": asdict(cfg),
            "architecture": {
                "name": "MDAVNet",
                "encoder": "resnet18",
                "encoder_weights": None,
                "rgb_channels": 3,
                "dct_bins": DCT_BINS,
                "dct_dim": DCT_DIM,
                "classes": 2,
                "jpeg_quality": JPEG_QUALITY,
                "stride": STRIDE,
                "mean": MEAN,
                "std": STD,
            },
            "best_epoch": epoch,
            "best_val": val_metrics,
            "dataset_root": str(DATA_ROOT),
            "metadata_rows_used": int(len(rows)),
        }
        torch.save(payload, best_path)
        print("saved new best:", best_path, "dice=", best_score)

with open(OUT / "training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)
print("Best checkpoint:", best_path)
print("Best validation dice:", best_score)

## 7. Visual sanity check

This cell overlays the predicted AI-edited region on validation samples. The prediction does not need to be perfect early in training, but after a real run it should light up around edited words rather than the whole document.

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def show_predictions(n: int = 4):
    model.eval()
    shown = 0
    for rgb, dct, mask, ids in val_loader:
        rgb = rgb.to(DEVICE)
        dct = dct.to(DEVICE)
        logits = model(rgb, dct)
        prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        rgb_cpu = rgb.cpu()
        for i in range(rgb_cpu.size(0)):
            img = rgb_cpu[i].clone()
            for c, (mean, std) in enumerate(zip(MEAN, STD)):
                img[c] = img[c] * std + mean
            img_np = np.clip(img.permute(1, 2, 0).numpy(), 0, 1)
            mask_np = mask[i].numpy()
            prob_np = prob[i]
            fig, ax = plt.subplots(1, 3, figsize=(12, 4))
            ax[0].imshow(img_np); ax[0].set_title(str(ids[i])); ax[0].axis("off")
            ax[1].imshow(mask_np, cmap="gray"); ax[1].set_title("ground truth mask"); ax[1].axis("off")
            ax[2].imshow(img_np); ax[2].imshow(prob_np, cmap="magma", alpha=0.55, vmin=0, vmax=1); ax[2].set_title("predicted AI edit probability"); ax[2].axis("off")
            plt.show()
            shown += 1
            if shown >= n:
                return

show_predictions(4)

## 8. Export contract and backend handoff

After training, download/copy:

```text
/kaggle/working/aiforge_diffusion_training/best_diffusion.pth
```

Place it in this repository as:

```text
models/best_diffusion.pth
```

Then set the backend environment variable:

```text
MDAV_DIFFUSION_WEIGHTS=../models/best_diffusion.pth
```

The checkpoint architecture and preprocessing match `backend/app/services/diffusion_service.py`: DCT bins `0..20`, DCT embedding dimension `16`, ResNet18 U-Net, 2 output classes.

In [ ]:
contract = f"""# AIForge Diffusion Model Contract

Checkpoint: `{best_path}`

## Backend consumer

`backend/app/services/diffusion_service.py`

## Architecture

- Model class: `MDAVNet`
- Segmentation backbone: `segmentation_models_pytorch.Unet`
- Encoder: `resnet18`
- Encoder weights: `None` at construction time; learned weights come from checkpoint
- Input channels: `3 + {DCT_DIM}`
- Output classes: `2`
- Class 0: authentic/background
- Class 1: AI-generated edited region

## Preprocessing

- Convert image to grayscale.
- Save temporary JPEG with quality `{JPEG_QUALITY}`.
- Read JPEG DCT coefficients with `jpegio`.
- Use `abs(coef)`, clipped to bins `0..{DCT_BINS - 1}`.
- Convert the JPEG pixels back to RGB.
- Normalize RGB with ImageNet mean `{MEAN}` and std `{STD}`.
- Pad image and DCT tensors to stride `{STRIDE}` during backend inference.

## Training data

- Dataset root: `{DATA_ROOT}`
- Metadata rows used: `{len(rows)}`
- Train records: `{len(train_records)}`
- Validation records: `{len(val_records)}`
- Clean authentic negatives found: `{num_clean}`

## Best validation

```json
{json.dumps({'best_val': history[-1]['val'] if history else None, 'best_score': best_score}, indent=2)}
```
"""

(OUT / "MODEL_CONTRACT.md").write_text(contract, encoding="utf-8")
print(contract)
print("Saved:", OUT / "MODEL_CONTRACT.md")

In [ ]:
# Convenience: copy the best checkpoint to /kaggle/working as well.
import shutil
shutil.copy2(best_path, Path("/kaggle/working") / cfg.checkpoint_name)
print("Copied to:", Path("/kaggle/working") / cfg.checkpoint_name)
print("Output files:")
for p in sorted(OUT.glob("*")):
    print(" -", p, round(p.stat().st_size / 1024 / 1024, 2), "MB")